# Stage 4. Multi-Task Training and Model Comparison

Notebook ini melatih model multi-task tiga head (regresi hemoglobin, klasifikasi anemia, severity ordinal) pada fitur palm memakai k-fold cross-validation berbasis pasien, membandingkan konfigurasi Path A saja, Path B saja, dan Full Fusion memakai src.common.train.run_kfold yang situs-agnostik. Baseline SVM pada fitur hand-crafted saja dijalankan lebih dulu untuk dibandingkan dengan literatur studi analog (Peksi dkk. 2021 sekitar 87.5-92.3 persen, Asare dkk. 2023 sekitar 98 persen), mengingat belum ada paper yang memodelkan video palm dataset ini secara langsung.

## Environment Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import numpy as np
import pandas as pd

from configs import paths
from src.common import features, manifest as manifest_utils, train

output_dir = paths.outputs_dir("palm")
manifest = pd.read_csv(output_dir / "manifest.csv")
manifest = manifest[manifest["roi_precropped"]].reset_index(drop=True)
manifest = manifest_utils.assign_kfold(manifest, n_splits=5, seed=42)

handcrafted = pd.read_csv(output_dir / "handcrafted_features.csv")
deep_embeddings = np.load(output_dir / "deep_embeddings.npy")
embedding_uids = pd.read_csv(output_dir / "deep_embeddings_uids.csv")["uid"].tolist()
print("sampel training", len(manifest))

## Sanity Check: Handcrafted Features with Classical SVM

SVM RBF pada fitur hand-crafted terstandardisasi sebagai baseline klasik sebelum model deep, memberi acuan performa minimum yang harus dilampaui oleh model multi-task fusi.

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

svm_features = StandardScaler().fit_transform(handcrafted.drop(columns=["uid"]))
svm_labels = manifest.set_index("uid").loc[handcrafted["uid"], "anemic"].to_numpy()
svm_scores = cross_val_score(SVC(kernel="rbf"), svm_features, svm_labels, cv=5, scoring="accuracy")
print("SVM accuracy per fold", np.round(svm_scores, 3))
print("SVM mean accuracy", round(svm_scores.mean(), 3))

## Model Configurations

Tiga konfigurasi jalur fitur dibandingkan dengan protokol pelatihan identik lewat parameter use_handcrafted dan use_deep pada run_kfold, sehingga perbandingan antar jalur adil.

In [ ]:
configurations = {
    "path_a_handcrafted": dict(use_handcrafted=True, use_deep=False),
    "path_b_deep": dict(use_handcrafted=False, use_deep=True),
    "full_fusion": dict(use_handcrafted=True, use_deep=True),
}

## Train and Evaluate All Configurations

In [ ]:
results = {}
for name, overrides in configurations.items():
    results[name] = train.run_kfold(
        manifest, handcrafted, deep_embeddings, embedding_uids,
        n_splits=5, epochs=60, **overrides,
    )
    fold_metrics = results[name]["fold_metrics"]
    print(name, "MAE", round(fold_metrics["mae"].mean(), 3), "accuracy", round(fold_metrics["accuracy"].mean(), 3))

## Compare Against Literature Baselines

Akurasi klasifikasi anemia dibandingkan terhadap studi analog yang memakai modalitas serupa (fingernail/palm), karena tidak ada paper yang memodelkan video palm dataset spesifik ini.

In [ ]:
literature_baselines = {
    "Peksi et al. 2021 (Naive Bayes, nail+palm)": (0.875, 0.923),
    "Asare et al. 2023 (CNN, fingernail+palm+conjunctiva)": (0.98, 0.98),
}
for name, result in results.items():
    accuracy = result["fold_metrics"]["accuracy"].mean()
    print(f"{name}: accuracy {accuracy:.3f}")
for label, (low, high) in literature_baselines.items():
    print(f"{label}: {low:.3f} - {high:.3f}")

## Save Results

In [ ]:
comparison_rows = []
for name, result in results.items():
    fold_metrics = result["fold_metrics"]
    comparison_rows.append({
        "configuration": name,
        "mae": fold_metrics["mae"].mean(),
        "rmse": fold_metrics["rmse"].mean(),
        "accuracy": fold_metrics["accuracy"].mean(),
        "severity_accuracy": fold_metrics["severity_accuracy"].mean(),
    })
comparison_table = pd.DataFrame(comparison_rows)
comparison_table.to_csv(output_dir / "multitask_model_comparison.csv", index=False)
results["full_fusion"]["oof"].to_csv(output_dir / "multitask_oof_full_fusion.csv", index=False)
comparison_table

## Hyperparameter Search with Optuna

Pencarian hyperparameter pada konfigurasi full fusion memakai Optuna, menyasar bobot loss, dimensi trunk, dan dropout, dengan objective MAE rata-rata k-fold.

In [ ]:
import optuna


def objective(trial):
    loss_weights = (
        trial.suggest_float("weight_regression", 0.5, 1.5),
        trial.suggest_float("weight_classification", 0.5, 1.5),
        trial.suggest_float("weight_severity", 0.1, 1.0),
    )
    trunk_dim = trial.suggest_categorical("trunk_dim", [64, 128, 256])
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    result = train.run_kfold(
        manifest, handcrafted, deep_embeddings, embedding_uids,
        n_splits=5, epochs=40, loss_weights=loss_weights,
        trunk_dim=trunk_dim, dropout=dropout,
    )
    return result["fold_metrics"]["mae"].mean()


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)
print("best params", study.best_params)
print("best MAE", study.best_value)

## Retrain with Best Hyperparameters

In [ ]:
best_params = study.best_params
best_loss_weights = (
    best_params["weight_regression"],
    best_params["weight_classification"],
    best_params["weight_severity"],
)
tuned_result = train.run_kfold(
    manifest, handcrafted, deep_embeddings, embedding_uids,
    n_splits=5, epochs=80, loss_weights=best_loss_weights,
    trunk_dim=best_params["trunk_dim"], dropout=best_params["dropout"],
)
tuned_metrics = tuned_result["fold_metrics"]
print("tuned MAE", round(tuned_metrics["mae"].mean(), 3), "tuned accuracy", round(tuned_metrics["accuracy"].mean(), 3))

## Updated Comparison and Save

In [ ]:
updated_rows = list(comparison_rows)
updated_rows.append({
    "configuration": "full_fusion_tuned",
    "mae": tuned_metrics["mae"].mean(),
    "rmse": tuned_metrics["rmse"].mean(),
    "accuracy": tuned_metrics["accuracy"].mean(),
    "severity_accuracy": tuned_metrics["severity_accuracy"].mean(),
})
comparison_table = pd.DataFrame(updated_rows)
comparison_table.to_csv(output_dir / "multitask_model_comparison.csv", index=False)
tuned_result["oof"].to_csv(output_dir / "multitask_oof_full_fusion_tuned.csv", index=False)
comparison_table

## Backbone Comparison: MobileNetV3-Small (Frozen Embedding)

Baseline sebelumnya hanya memakai ResNet18-CSA sebagai backbone tanpa dibandingkan dengan alternatif lain pada data palm sendiri. Sel ini mengekstrak embedding dengan backbone MobileNetV3-Small (arsitektur yang lebih ringan, cocok untuk deployment edge), lalu membandingkan Path B saja dan Full Fusion terhadap hasil ResNet18-CSA dengan protokol identik, mengikuti metodologi yang sama seperti conjunctiva stage 5. Embedding disimpan ke file terpisah agar tidak menimpa deep_embeddings.npy milik ResNet18.

In [ ]:
mobilenet_backbone = features.EmbeddingBackbone(backbone_name="mobilenet_v3_small")
deep_embeddings_mobilenet, embedding_uids_mobilenet = features.extract_deep_embeddings(manifest, model=mobilenet_backbone)
np.save(output_dir / "deep_embeddings_mobilenet.npy", deep_embeddings_mobilenet)
pd.DataFrame({"uid": embedding_uids_mobilenet}).to_csv(output_dir / "deep_embeddings_mobilenet_uids.csv", index=False)
print("mobilenet embedding shape", deep_embeddings_mobilenet.shape)

mobilenet_configurations = {
    "path_b_deep_mobilenet": dict(use_handcrafted=False, use_deep=True),
    "full_fusion_mobilenet": dict(use_handcrafted=True, use_deep=True),
}
mobilenet_results = {}
for name, overrides in mobilenet_configurations.items():
    mobilenet_results[name] = train.run_kfold(
        manifest, handcrafted, deep_embeddings_mobilenet, embedding_uids_mobilenet,
        n_splits=5, epochs=60, **overrides,
    )
    fold_metrics = mobilenet_results[name]["fold_metrics"]
    print(name, "MAE", round(fold_metrics["mae"].mean(), 4), "accuracy", round(fold_metrics["accuracy"].mean(), 4))

## Severity Class Weighting (Frozen Embedding, Full Fusion ResNet18)

Kelas Moderate hanya 72 dari 809 sampel (8.9 persen), dan CornLoss ordinal sebelumnya tidak berbobot kelas, sehingga model cenderung tidak pernah memprediksi kelas ini. Sel ini menjalankan ulang konfigurasi full_fusion (ResNet18) dengan weight_severity_classes aktif, dibandingkan langsung terhadap hasil full_fusion tanpa bobot kelas yang sudah ada, memakai kappa sebagai metrik utama karena akurasi saja tidak sensitif terhadap kelas minoritas yang diabaikan.

In [ ]:
from sklearn.metrics import cohen_kappa_score

weighted_severity_result = train.run_kfold(
    manifest, handcrafted, deep_embeddings, embedding_uids,
    n_splits=5, epochs=60, use_handcrafted=True, use_deep=True, weight_severity_classes=True,
)


def _severity_kappa(oof):
    severity_valid = oof["severity_true"] >= 0
    if not severity_valid.any():
        return float("nan")
    return cohen_kappa_score(oof.loc[severity_valid, "severity_true"], oof.loc[severity_valid, "severity_pred"])


baseline_kappa = _severity_kappa(results["full_fusion"]["oof"])
weighted_kappa = _severity_kappa(weighted_severity_result["oof"])
print("full_fusion kappa (tanpa bobot kelas)", round(baseline_kappa, 4))
print("full_fusion kappa (dengan bobot kelas severity)", round(weighted_kappa, 4))

weighted_severity_result["oof"].to_csv(output_dir / "multitask_oof_full_fusion_weighted_severity.csv", index=False)

## Consolidated Comparison and Save

Semua konfigurasi frozen-embedding (ResNet18 asli, ResNet18 dengan bobot kelas severity, MobileNetV3-Small) digabung ke satu tabel yang sama, ditambah kappa severity untuk setiap baris, sehingga keputusan backbone dan efek bobot kelas dapat dibaca dari satu tempat.

In [ ]:
all_named_results = dict(results)
all_named_results["full_fusion_tuned"] = tuned_result
all_named_results["full_fusion_weighted_severity"] = weighted_severity_result
all_named_results.update(mobilenet_results)

final_rows = []
for name, result in all_named_results.items():
    fold_metrics = result["fold_metrics"]
    final_rows.append({
        "configuration": name,
        "mae": fold_metrics["mae"].mean(),
        "rmse": fold_metrics["rmse"].mean(),
        "accuracy": fold_metrics["accuracy"].mean(),
        "severity_accuracy": fold_metrics["severity_accuracy"].mean(),
        "severity_kappa": _severity_kappa(result["oof"]),
    })

comparison_table = pd.DataFrame(final_rows)
comparison_table.to_csv(output_dir / "multitask_model_comparison.csv", index=False)
comparison_table.round(4)